# SK하이닉스 종합 분석

주가, 네이버 뉴스, 증권사 리포트(KIS → 네이버 증권 → 한경컨센서스), LLM 분석을 실행합니다.

In [1]:
# 필요한 패키지: requests, pandas, python-dotenv
import os
import datetime as dt
import re
import requests
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from email.utils import parsedate_to_datetime
from html import unescape
from IPython.display import display

_dotenv_paths = []
for _base_path in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    _dotenv_paths.extend([
        _base_path / ".env",
        _base_path / "notebooks" / ".env",
    ])
if "__file__" in globals():
    _dotenv_paths.extend([
        Path(__file__).resolve().with_name(".env"),
        Path(__file__).resolve().parents[1] / "notebooks" / ".env",
    ])
for _dotenv_path in _dotenv_paths:
    if _dotenv_path.is_file():
        load_dotenv(_dotenv_path, override=True)

STOCK_NAME = "GST"
STOCK_TICKER = "083450"
LOOKBACK_DAYS = 30
NEWS_COUNT = 20

# 공식 KRX Open API(data-dbg.krx.co.kr)의 일별매매정보 엔드포인트만 사용한다.
# PER/PBR/배당수익률은 공식 API에 없고 data.krx.co.kr 비공식 스크래핑 경로뿐이라
# 이번 리라이트에서는 다루지 않는다 (HANDOFF.md의 제품 범위 결정을 따름).
KRX_BASE_URL = "https://data-dbg.krx.co.kr/svc/apis/sto"
KRX_MARKET_PATHS = ("stk", "ksq")  # 코스피, 코스닥


In [2]:
def _clean_html(value: str) -> str:
    return unescape(value.replace("<b>", "").replace("</b>", "")).strip()


def _krx_auth_key() -> str:
    key = os.getenv("KRX_AUTH_KEY")
    if not key:
        raise RuntimeError(".env에 KRX_AUTH_KEY를 설정하세요.")
    return key


def _fetch_krx_day(market_path: str, bas_dd: str, key: str) -> list:
    """공식 KRX 일별매매정보 API에서 특정 날짜의 전체 종목 스냅샷을 가져온다."""
    response = requests.get(
        f"{KRX_BASE_URL}/{market_path}_bydd_trd",
        params={"AUTH_KEY": key, "basDd": bas_dd},
        timeout=20,
    )
    if not response.ok:
        raise RuntimeError(f"KRX {market_path} API 오류 ({response.status_code}): {response.text}")
    return response.json().get("OutBlock_1") or []


def _collect_krx_rows(ticker: str, days: int, market_path: str, key: str) -> list:
    rows = []
    date = dt.date.today()
    checked = 0
    max_checked = days * 2 + 10  # 주말·공휴일을 감안해 넉넉히 조회
    while len(rows) < days and checked < max_checked:
        day_rows = _fetch_krx_day(market_path, date.strftime("%Y%m%d"), key)
        row = next((item for item in day_rows if item.get("ISU_CD") == ticker), None)
        if row:
            rows.append(row)
        date -= dt.timedelta(days=1)
        checked += 1
    return rows


def get_stock_history(ticker: str, days: int = 30) -> pd.DataFrame:
    """공식 KRX 일별매매정보 API로 최근 영업일 주가·거래량을 조회한다.

    data.krx.co.kr을 스크래핑하는 pykrx 대신 KRX_AUTH_KEY로 인증하는 공식 API만 사용한다
    (비공식 스크래핑을 쓰지 않는다는 프로젝트 방침, HANDOFF.md 참고).
    """
    key = _krx_auth_key()
    for market_path in KRX_MARKET_PATHS:
        rows = _collect_krx_rows(ticker, days, market_path, key)
        if rows:
            frame = pd.DataFrame(rows)
            frame["날짜"] = pd.to_datetime(frame["BAS_DD"])
            frame = frame.set_index("날짜").sort_index()
            return pd.DataFrame({
                "시가": frame["TDD_OPNPRC"].astype(float),
                "고가": frame["TDD_HGPRC"].astype(float),
                "저가": frame["TDD_LWPRC"].astype(float),
                "종가": frame["TDD_CLSPRC"].astype(float),
                "거래량": frame["ACC_TRDVOL"].astype(float).astype("int64"),
                "등락률": frame["FLUC_RT"].astype(float),
            })
    raise RuntimeError(f"{ticker} 종목의 주가 데이터를 찾지 못했습니다.")


def search_stock_news(query: str, display: int = 20) -> pd.DataFrame:
    """네이버 뉴스 검색 API에서 최신 관련 기사를 가져온다."""
    client_id = os.getenv("NAVER_CLIENT_ID")
    client_secret = os.getenv("NAVER_CLIENT_SECRET")
    if not client_id or not client_secret:
        raise RuntimeError(".env에 NAVER_CLIENT_ID와 NAVER_CLIENT_SECRET을 설정하세요.")

    response = requests.get(
        "https://openapi.naver.com/v1/search/news.json",
        headers={
            "X-Naver-Client-Id": client_id,
            "X-Naver-Client-Secret": client_secret,
        },
        params={"query": query, "display": display, "sort": "date"},
        timeout=20,
    )
    if not response.ok:
        raise RuntimeError(f"네이버 뉴스 API 오류 ({response.status_code}): {response.text}")

    rows = []
    for item in response.json().get("items", []):
        published_at = item.get("pubDate")
        rows.append({
            "title": _clean_html(item.get("title", "")),
            "description": _clean_html(item.get("description", "")),
            "link": item.get("link", ""),
            "pubDate": (
                parsedate_to_datetime(published_at)
                if published_at
                else None
            ),
        })
    return pd.DataFrame(rows)


def analyze_with_llm(stock_name: str, ticker: str, history: pd.DataFrame, news: pd.DataFrame) -> str:
    """주가 흐름과 뉴스의 관계를 xAI LLM에 분석시킨다."""
    api_key = os.getenv("XAI_API_KEY")
    if not api_key:
        raise RuntimeError(".env에 XAI_API_KEY를 설정하세요.")

    price = history.copy().reset_index()
    price_text = price.tail(15).to_string(index=False)
    news_text = (
        news[["title", "description", "pubDate", "link"]].to_string(index=False)
        if not news.empty
        else "관련 뉴스 없음"
    )

    system = """너는 한국 주식 리서치 애널리스트다. 제공된 데이터만 근거로 분석하고, 확인된 사실과 해석을 구분하라.
투자 매수·매도 권유나 확정적인 미래 예측은 하지 말라."""
    prompt = f"""다음은 {stock_name}({ticker})의 최근 주가와 네이버 뉴스다.

[주가 데이터]
{price_text}

[관련 뉴스]
{news_text}

아래 형식으로 한국어 종합 분석을 작성해라.
1. 최근 주가 흐름: 기간 수익률, 고점·저점, 거래량 변화
2. 핵심 뉴스 요약: 주가에 영향을 줄 수 있는 뉴스 3~5개
3. 주가 변동 원인: 뉴스와 주가·거래량의 시간적 흐름을 연결한 근거 중심 분석
4. 긍정 요인과 부정 요인
5. 추가 확인할 리스크와 다음 거래일에 관찰할 지표
각 항목은 간결한 문단 또는 bullet로 작성하고, 근거가 부족하면 '판단 유보'라고 표시해라."""

    response = requests.post(
        "https://api.x.ai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": os.getenv("XAI_MODEL", "grok-4-1-fast-non-reasoning"),
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
            ],
            "temperature": 0.2,
        },
        timeout=120,
    )
    if not response.ok:
        raise RuntimeError(f"xAI API 오류 ({response.status_code}): {response.text}")
    return response.json()["choices"][0]["message"]["content"]


In [3]:
price_df = get_stock_history(STOCK_TICKER, LOOKBACK_DAYS)
news_df = search_stock_news(STOCK_NAME, NEWS_COUNT)

first_close = float(price_df["종가"].iloc[0])
last_close = float(price_df["종가"].iloc[-1])
period_return = (last_close / first_close - 1) * 100
print(f"종목: {STOCK_NAME} ({STOCK_TICKER})")
print(f"조회기간: {price_df.index.min().date()} ~ {price_df.index.max().date()}")
print(f"최근 종가: {last_close:,.0f}원 | 기간 수익률: {period_return:+.2f}%")
print(f"수집 뉴스: {len(news_df)}건")
display(price_df.tail(10))
display(news_df.head(10))

llm_report = analyze_with_llm(STOCK_NAME, STOCK_TICKER, price_df, news_df)
print()
print("===== LLM 종합 분석 =====")
print()
print(llm_report)


종목: GST (083450)
조회기간: 2026-07-14 ~ 2026-08-26
최근 종가: 44,800원 | 기간 수익률: -3.14%
수집 뉴스: 20건


,시가,고가,저가,종가,거래량,등락률
날짜,,,,,,
2026-08-12,50700.0,51500.0,49350.0,50900.0,133636,0.20
2026-08-13,51600.0,52500.0,50200.0,50500.0,114888,-0.79
2026-08-14,50900.0,51300.0,48250.0,49300.0,123268,-2.38
2026-08-18,49950.0,50500.0,46350.0,47100.0,244371,-4.46
2026-08-19,45100.0,47500.0,44400.0,45950.0,129291,-2.44
2026-08-20,46450.0,47000.0,45400.0,45850.0,83920,-0.22
2026-08-21,44450.0,44800.0,42900.0,43000.0,113287,-6.22
2026-08-24,43000.0,46200.0,42900.0,44650.0,153560,3.84
2026-08-25,43750.0,45550.0,41100.0,45050.0,160873,0.90


,title,description,link,pubDate
0,HBM4 넘어 첨단 패키징까지…반도체 장비주 상승세 확산,"세정·열처리와 소재 분야에서는 저스템, 싸이맥스, 예스티, 티씨케이, GST, 코미...",https://www.pinpointnews.co.kr/news/articleVie...,2026-08-27 15:32:00+09:00
1,HBM과 후공정 장비주 동반 강세… 반도체 업종 전반 강한 반등 장세 주도,"지앤비에스 에코, 칩스앤미디어, ISC, 세미파이브, 가온칩스, 티에스이, 뉴파워프...",https://www.pinpointnews.co.kr/news/articleVie...,2026-08-27 09:10:00+09:00
2,"한국, 세계 라면 소비 '9위'…중국 압도적 '1위'",인도는 상품서비스세(GST) 인하와 전자상거래 확대에 힘입어 전년 대비 8.9% 증...,https://www.theguru.co.kr/news/article.html?no...,2026-08-26 08:36:00+09:00
3,"씨엔티테크, 지역 스타트업 19개사와 대·중견기업 협업 논의",NEST GST 오픈이노베이션 밋업 데이'를 개최했다. 이번 행사에는 네이버클라우드...,https://www.shinailbo.co.kr/news/articleView.h...,2026-08-25 18:10:00+09:00
4,"[리스트] MVP 상위 20선...달바글로벌, GST 등 - 25일","달바글로벌 GST 등이 포함됐다고 밝혔다. 이에 따르면 이날 수급, 밸류에이션, 펀...",https://www.itooza.com/common/iview.php?no=202...,2026-08-25 07:22:00+09:00
5,"美 파리협정 탈퇴 틈타… 中, 2028년 유엔 기후총회 유치 추진",파리협정에 따라 각국의 기후대응이 목표대로 가고 있는지를 5년마다 평가하는 두 번째...,http://www.impacton.net/news/articleView.html?...,2026-08-24 11:04:00+09:00
6,"[기획] 고팍스, 2분기 상장 5개·상장 폐지 18개","상장 문턱을 높인 고팍스는 동시에 블로서리(BLY), 크레타(CRETA), 파우터(...",https://www.nbntv.co.kr/news/articleView.html?...,2026-08-24 10:32:00+09:00
7,"신보·충남콘텐츠진흥원 뭉쳤다…씨엔티테크, 대기업-스타트업 잇는 '오...",NEST GST 오픈이노베이션 밋업 데이'를 지난 19일 신용보증기금 광진센터에서 ...,https://n.news.naver.com/mnews/article/030/000...,2026-08-24 07:59:00+09:00
8,"[리스트] MVP 상위 주식 20선...인바디, 피에스케이 등 - 24일","화신정공, GST, 대덕전자, 샘씨엔에스, 티디에스팜 등이 포함됐다. 한편 아이투자...",https://www.itooza.com/common/iview.php?no=202...,2026-08-24 07:22:00+09:00
9,"충남콘진원, GST 입주기업 오픈이노베이션 밋업",충남콘텐츠진흥원(이하 충남콘진원)은 천안 그린스타트업타운(이하 GST) 입주기업과 ...,https://n.news.naver.com/mnews/article/656/000...,2026-08-23 12:24:00+09:00



===== LLM 종합 분석 =====

1. 최근 주가 흐름  
• 8월 5일~26일 기간 수익률: –5.8% (47,600 → 44,800원).  
• 고점 53,400원(8월 11일), 저점 41,100원(8월 25일).  
• 거래량: 8월 7일 31만 주로 급증한 뒤 8월 18~19일 24만·12만 주로 다시 확대됐고, 이후 7~16만 주 수준으로 축소.  
• 8월 21일 –6.22% 낙폭 이후 24~26일 3거래일 연속 소폭 반등했으나, 8월 11일 고점 대비 16% 하회.

2. 핵심 뉴스 요약  
• 8월 27일: “HBM4·첨단 패키징 수혜주”로 GST가 세정·열처리 장비군에 포함(핀포인트뉴스).  
• 8월 27일: “HBM·후공정 장비 동반 강세” 기사에서 GST가 우상향 종목으로 언급.  
• 8월 25일: 아이투자 MVP 상위 20선에 GST 포함(수급·밸류에이션·펀더멘털 80점 이상).  
• 8월 19일: 기관 매매 동향에서 GST가 매도세 종목으로 분류(핀포인트뉴스).  
• 8월 21일: 기계 업체 부채비율 비교 기사에서 GST 32.2%로 상위권(그린이코노믹).

3. 주가 변동 원인  
• 8월 7일 거래량 급증과 5.65% 상승은 8월 5~6일 반도체 장비군 강세 기사와 시기적으로 일치.  
• 8월 18~21일 3거래일 연속 하락(–4.46%→–2.44%→–6.22%)은 8월 19일 기관 매도 기사와 8월 21일 부채비율 기사가 나온 시점과 겹침.  
• 8월 24~26일 반등 구간에는 8월 25일 MVP 상위 편입 기사가 있었으나, 거래량은 오히려 감소해 수급 강도는 확인되지 않음.  
• 8월 27일 반도체 장비군 재부각 기사는 26일 종가 이후에 나온 것으로, 당일 주가에 반영된 증거는 없음.

4. 긍정 요인과 부정 요인  
긍정  
• HBM·후공정 장비 테마에서 반복적으로 언급돼 모멘텀 유지 가능성.  
• MVP 모델에서 수급·펀더멘털 동반 양호 평가.  

부정  
• 8월 19일 기관 매도 관측.  
•